# 01 · Ontology and Knowledge Graph Foundations

> Part of the **ICAPS 2026 Planning Ontology Tutorial**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai4society/ICAPS26-planning-ontology-tutorial/blob/main/notebooks/01_ontology_and_kg.ipynb)

The **planning ontology** gives automated planning a shared, machine-readable
vocabulary. In this notebook you put it to work end to end:

- **Load** the ontology straight from its persistent identifier (PURL)
- **Explore** its classes and properties
- **Build** a knowledge graph from a blocksworld PDDL pair
- **Query** that graph with SPARQL

> **Tip.** The **Core** path covers the essentials. **Go deeper** adds the prebuilt
> tutorial KG and a slice of the real IPC-2018 data.

---

## Setup

Run this first. In **Colab** it installs the dependencies and fetches the tutorial
data. **Locally** it uses your `requirements.txt` environment.

In [1]:
# In Colab this installs dependencies and fetches the tutorial data.
# Locally it assumes you installed requirements.txt.
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    subprocess.run(["pip", "install", "-q", "rdflib", "pandas"], check=True)
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/ai4society/ICAPS26-planning-ontology-tutorial.git"], check=False)
    DATA = Path("ICAPS26-planning-ontology-tutorial/data")
else:
    DATA = Path("..") / "data"

print("data directory:", DATA.resolve())

data directory: /Users/nitingupta/usc/ai4s/conferences/icaps26/tutorial/planning-ontology-tutorial/data


---
## 1. Load the ontology from its PURL

The ontology has a **persistent identifier** (PURL) that resolves to the current
release. RDFLib follows the redirects and parses the RDF/XML, so your code stays
stable across versions and hosting locations.

In [2]:
from rdflib import Graph, Namespace, Literal, RDF, RDFS, XSD
from rdflib.namespace import OWL

PO = Namespace("https://purl.org/ai4s/ontology/planning#")

onto = Graph()
onto.parse("https://purl.org/ai4s/ontology/planning", format="xml")
print(f"Loaded ontology: {len(onto)} triples")

Loaded ontology: 228 triples


---
## 2. Explore the schema

The ontology supplies the vocabulary for planning: classes such as
`PlanningDomain`, `Planner`, and `Action`, and the object and datatype properties
that connect them.

> **Note.** A few classes (`Plan`, `Step`, `Action`) build on established
> vocabularies (DUL, P-Plan, CARESSES), which is good ontology practice. The helper
> below lists the terms minted in the planning namespace.

In [3]:
import pandas as pd

def local_terms(graph, rdf_type):
    return sorted(str(s).split("#")[-1] for s in graph.subjects(RDF.type, rdf_type)
                  if str(s).startswith(str(PO)))

classes = local_terms(onto, OWL.Class)
obj_props = local_terms(onto, OWL.ObjectProperty)
data_props = local_terms(onto, OWL.DatatypeProperty)
print(f"{len(classes)} classes, {len(obj_props)} object properties, "
      f"{len(data_props)} datatype properties")

pd.DataFrame({"object properties": pd.Series(obj_props),
              "datatype properties": pd.Series(data_props)})

14 classes, 26 object properties, 4 datatype properties


,object properties,datatype properties
0,addsPredicate,hasActionExplanation
1,deletesPredicate,hasModelReconciliationExplanation
2,hasAction,hasPlanCost
3,hasConstant,hasPlanExplanation
4,hasEffect,NaN
5,hasExtractedAction,NaN
6,hasGoalState,NaN
7,hasHighRelevancePlanner,NaN
8,hasInitialState,NaN
9,hasLowRelevancePlanner,NaN


---
## 3. Build a knowledge graph from PDDL (Core)

A knowledge graph is the ontology **populated with data**. You read a blocksworld
domain and problem, extract the requirements, actions, and objects, and add them as
triples under the `po:` namespace.

> **Tip.** A light regular-expression parse keeps the focus on the mapping from
> PDDL to triples. For production pipelines the repo provides full extraction
> scripts.

In [4]:
import re

domain_text = (DATA / "domains" / "blocksworld" / "domain.pddl").read_text()
problem_text = (DATA / "domains" / "blocksworld" / "problem.pddl").read_text()

def parse_requirements(text):
    m = re.search(r":requirements([^)]*)\)", text)
    return re.findall(r":([\w-]+)", m.group(1)) if m else []

def parse_actions(text):
    out = {}
    for name, params in re.findall(r"\(:action\s+(\S+)\s+:parameters\s*\(([^)]*)\)", text):
        out[name] = re.findall(r"\?(\w+)\s*-\s*(\w+)", params)
    return out

requirements = parse_requirements(domain_text)
actions = parse_actions(domain_text)
print("requirements:", requirements)
print("actions:", list(actions))

requirements: ['strips']
actions: ['pick-up', 'put-down', 'stack', 'unstack']


In [5]:
kg = Graph()
kg.bind("po", PO)
kg.bind("rdfs", RDFS)

# Domain and its requirements
kg.add((PO.blocksworld, RDF.type, PO.PlanningDomain))
kg.add((PO.blocksworld, RDFS.label, Literal("blocksworld")))
for r in requirements:
    req = PO[r]
    kg.add((req, RDF.type, PO.DomainRequirement))
    kg.add((req, RDFS.label, Literal(":" + r)))
    kg.add((PO.blocksworld, PO.hasRequirement, req))

# Actions and their parameters
for name, params in actions.items():
    action = PO[name]
    kg.add((action, RDFS.label, Literal(name)))
    kg.add((PO.blocksworld, PO.hasAction, action))
    for var, typ in params:
        par = PO[f"{name}_{var}"]
        kg.add((par, RDFS.label, Literal(f"?{var} - {typ}")))
        kg.add((action, PO.hasParameter, par))

# Problem objects
m = re.search(r":objects([^)]*)\)", problem_text)
objects = re.findall(r"(\w+)", m.group(1).split("-")[0]) if m else []
kg.add((PO.problem_3_1, RDF.type, PO.PlanningProblem))
kg.add((PO.blocksworld, PO.hasProblem, PO.problem_3_1))
for o in objects:
    kg.add((PO[o], RDF.type, PO.ProblemObject))
    kg.add((PO.problem_3_1, PO.hasObject, PO[o]))

print(f"built KG: {len(kg)} triples, objects = {objects}")

built KG: 33 triples, objects = ['b1', 'b2', 'b3']


---
## 4. Query the KG with SPARQL (Core)

**SPARQL** is the query language for RDF. Declare your prefixes once, then ask the
graph questions.

In [6]:
PREFIX = (
    "PREFIX po: <https://purl.org/ai4s/ontology/planning#> "
    "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> "
)

print("Actions in blocksworld:")
for row in kg.query(PREFIX + """
    SELECT ?label WHERE { po:blocksworld po:hasAction ?a . ?a rdfs:label ?label }
    ORDER BY ?label"""):
    print(" ", row.label)

print("\nRequirements:")
for row in kg.query(PREFIX + """
    SELECT ?label WHERE { po:blocksworld po:hasRequirement ?r . ?r rdfs:label ?label }"""):
    print(" ", row.label)

Actions in blocksworld:
  pick-up
  put-down
  stack
  unstack

Requirements:
  :strips


> **Tip.** Serialize the graph to Turtle at any point to inspect or share it.

In [7]:
print(kg.serialize(format="turtle")[:700])

@prefix po: <https://purl.org/ai4s/ontology/planning#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

po:blocksworld a po:PlanningDomain ;
    rdfs:label "blocksworld" ;
    po:hasAction po:pick-up,
        po:put-down,
        po:stack,
        po:unstack ;
    po:hasProblem po:problem_3_1 ;
    po:hasRequirement po:strips .

po:b1 a po:ProblemObject .

po:b2 a po:ProblemObject .

po:b3 a po:ProblemObject .

po:pick-up rdfs:label "pick-up" ;
    po:hasParameter po:pick-up_x .

po:pick-up_x rdfs:label "?x - block" .

po:problem_3_1 a po:PlanningProblem ;
    po:hasObject po:b1,
        po:b2,
        po:b3 .

po:put-down rdfs:label "put-down" ;
    po:hasParameter po:put-down_x .


---
## Go deeper: the prebuilt tutorial KG and real IPC-2018 data

The KG you just built captures the domain's **structure**. The repo also ships a
fuller tutorial KG with planner relevance, a plan, and explanations (used by
notebooks 02 and 03), alongside a slice of the **real IPC-2018 knowledge graph**.

> **Note.** The blocksworld relevance values here are curated for the tutorial.
> Notebook 02 runs the same queries against the real IPC-2018 data.

In [8]:
tutorial_kg = Graph()
tutorial_kg.parse(str(DATA / "kgs" / "blocksworld_tutorial.ttl"), format="turtle")
print(f"prebuilt tutorial KG: {len(tutorial_kg)} triples")

print("\nPlanner relevance for blocksworld:")
for row in tutorial_kg.query(PREFIX + """
    SELECT ?bucket ?label WHERE {
      VALUES (?p ?bucket) {
        (po:hasHighRelevancePlanner "high") (po:hasMediumRelevancePlanner "medium")
        (po:hasLowRelevancePlanner "low") }
      po:blocksworld ?p ?planner . ?planner rdfs:label ?label }"""):
    print(f"  {row.bucket:<6} {row.label}")

prebuilt tutorial KG: 106 triples

Planner relevance for blocksworld:
  high   Fast Downward
  high   LAMA-2011
  medium BFWS
  low    Probe


> **Working across ontology versions.** The published v2.0 ontology and the
> IPC-2018 instance data share the same property names (`hasHighRelevancePlanner`).
> The instance data was released under an earlier namespace, so you point SPARQL at
> that version's prefix, and RDFLib loads either one the same way. The full IPC-2018
> graph is ~24 MB, so this repo ships a compact two-domain slice.

In [9]:
OLD = "http://www.semanticweb.org/muppa/ontologies/2022/4/plan-ontology#"

real = Graph()
real.parse(str(DATA / "kgs" / "ipc2018_planner_info_sample.ttl"), format="turtle")
print(f"real IPC-2018 subset: {len(real)} triples")

for row in real.query(f"""
    PREFIX po: <{OLD}>
    SELECT ?domain (COUNT(?planner) AS ?n) WHERE {{
        ?domain po:hasHighRelevancePlanner ?planner }}
    GROUP BY ?domain"""):
    print(f"  {str(row.domain).split('#')[-1]}: {int(row.n)} high-relevance planners")

# The full graph loads the same way, straight from the repo (large):
# real.parse("https://raw.githubusercontent.com/ai4society/planning-ontology/main/"
#            "AI-Planning-Ontology/models/plan-ontology-rdf-instances-planner-info_ipc2018.owl",
#            format="xml")

real IPC-2018 subset: 56 triples
  agricola: 5 high-relevance planners
  caldera: 8 high-relevance planners


---
## Recap and next

You loaded the ontology from its PURL, explored the schema, populated a knowledge
graph from PDDL, and queried it with SPARQL. You also met the prebuilt tutorial KG
and the real IPC-2018 data.

| Next notebook | Focus |
| --- | --- |
| **02 · Planner selection** | Rank the planners for a domain |
| **03 · Plan explanation** | Turn a plan into a readable narrative |